# Netflix Member Retention — Initial EDA & Baseline Model
**Assignment 20.1 | Berkeley AI/ML Capstone**

## Research Question
Which member behaviors and household attributes most strongly predict Netflix retention (subscription tenure / churn risk)?

## Dataset
Simulated dataset of 7,500 member records with streaming behavior, engagement metrics, and household demographics. The target variable is `churned` (1 = member churned within the observation window, 0 = retained).

**Note:** All data in this project is synthetically generated for academic purposes. It does not represent or contain any real Netflix member data.

---

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 2. Load Data

In [ ]:
df = pd.read_csv('data/netflix_retention_simulated.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 3. Data Cleaning

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'No missing values found.')

In [ ]:
# Check for duplicates
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')

In [ ]:
# Sanity check: titles_completed should never exceed titles_started
bad_rows = df[df['titles_completed'] > df['titles_started']]
print(f'Rows where completed > started: {len(bad_rows)}')

# Completion rate should be between 0 and 1
print(f'Completion rate range: {df["completion_rate"].min():.2f} - {df["completion_rate"].max():.2f}')

In [ ]:
# Outlier check on view_hours_30d using IQR
Q1 = df['view_hours_30d'].quantile(0.25)
Q3 = df['view_hours_30d'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['view_hours_30d'] < Q1 - 1.5*IQR) | (df['view_hours_30d'] > Q3 + 1.5*IQR)]
print(f'Outliers in view_hours_30d: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')
# These are extreme but plausible viewers — keeping them in

Data looks clean. No missing values, no duplicates, and logical constraints are satisfied. The outliers in viewing hours represent heavy users, which are plausible and meaningful for the analysis — keeping them.

## 4. Exploratory Data Analysis

### 4.1 Target Variable: Churn Distribution

In [ ]:
churn_counts = df['churned'].value_counts()
churn_pct = df['churned'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(['Retained', 'Churned'], churn_counts.values, color=['steelblue', 'salmon'])
axes[0].set_title('Member Churn Count')
axes[0].set_ylabel('Number of Members')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=['Retained', 'Churned'],
            autopct='%1.1f%%', colors=['steelblue', 'salmon'], startangle=90)
axes[1].set_title('Churn Rate')

plt.suptitle(f'Overall Churn Rate: {churn_pct:.1f}%', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.2 Churn by Region and Device Mix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Churn rate by region
region_churn = df.groupby('region')['churned'].mean().sort_values(ascending=False)
axes[0].barh(region_churn.index, region_churn.values * 100, color='steelblue')
axes[0].set_xlabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Region')
for i, v in enumerate(region_churn.values):
    axes[0].text(v*100 + 0.2, i, f'{v*100:.1f}%', va='center')

# Churn rate by device mix
device_churn = df.groupby('device_mix')['churned'].mean().sort_values(ascending=False)
axes[1].barh(device_churn.index, device_churn.values * 100, color='coral')
axes[1].set_xlabel('Churn Rate (%)')
axes[1].set_title('Churn Rate by Device Mix')
for i, v in enumerate(device_churn.values):
    axes[1].text(v*100 + 0.2, i, f'{v*100:.1f}%', va='center')

plt.tight_layout()
plt.show()

### 4.3 Churn by Kids in Household

In [ ]:
kids_churn = df.groupby('num_kids')['churned'].mean()

plt.figure(figsize=(7, 4))
plt.bar(kids_churn.index, kids_churn.values * 100, color='mediumpurple')
plt.xlabel('Number of Kids in Household')
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Number of Kids in Household')
for i, v in enumerate(kids_churn.values):
    plt.text(kids_churn.index[i], v*100 + 0.2, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.show()

### 4.4 Continuous Features vs Churn

In [ ]:
numeric_features = ['view_hours_30d', 'completion_rate', 'active_days_30d',
                    'titles_started', 'my_list_saves', 'kids_content_hours']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    retained = df[df['churned'] == 0][col]
    churned = df[df['churned'] == 1][col]
    axes[i].hist(retained, bins=30, alpha=0.6, label='Retained', color='steelblue')
    axes[i].hist(churned, bins=30, alpha=0.6, label='Churned', color='salmon')
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].legend(fontsize=8)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions: Retained vs Churned Members', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.5 Correlation Heatmap

In [ ]:
corr_cols = ['view_hours_30d', 'completion_rate', 'active_days_30d',
             'titles_started', 'titles_completed', 'thumbs_up',
             'my_list_saves', 'num_kids', 'num_profiles', 'churned']

plt.figure(figsize=(10, 8))
corr_matrix = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Matrix — Member Features vs Churn', fontsize=12)
plt.tight_layout()
plt.show()

### 4.6 Tenure Distribution by Churn Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(data=df, x='churned', y='tenure_months', ax=axes[0],
            palette={0: 'steelblue', 1: 'salmon'})
axes[0].set_xticklabels(['Retained', 'Churned'])
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_ylabel('Tenure (Months)')

sns.boxplot(data=df, x='churned', y='active_days_30d', ax=axes[1],
            palette={0: 'steelblue', 1: 'salmon'})
axes[1].set_xticklabels(['Retained', 'Churned'])
axes[1].set_title('Active Days (Last 30d) by Churn Status')
axes[1].set_ylabel('Active Days')

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Create derived features
df['has_kids'] = (df['num_kids'] > 0).astype(int)
df['is_mobile_only'] = (df['device_mix'] == 'mobile_only').astype(int)
df['is_multi_device'] = (df['device_mix'] == 'multi_device').astype(int)

# Engagement intensity: how engaged per active day
df['hours_per_active_day'] = (df['view_hours_30d'] / df['active_days_30d'].replace(0, 1)).round(2)

# Activity regularity: active days as a fraction of the month
df['activity_rate'] = (df['active_days_30d'] / 30).round(3)

# Social engagement: combined thumbs + saves
df['social_engagement'] = df['thumbs_up'] + df['my_list_saves']

print('New features added:')
new_cols = ['has_kids', 'is_mobile_only', 'is_multi_device',
            'hours_per_active_day', 'activity_rate', 'social_engagement']
print(df[new_cols].describe().round(2))

In [ ]:
# One-hot encode region
df = pd.get_dummies(df, columns=['region'], drop_first=True)
print('Columns after encoding:', df.shape[1])

## 6. Baseline Model: Logistic Regression

I'll use logistic regression as the baseline. It's interpretable and appropriate for binary classification. The evaluation metric I'll focus on is **ROC-AUC**, which measures how well the model separates churners from retained members — important here because we care about ranking risk, not just raw accuracy.

In [ ]:
# Select features for the model
feature_cols = [
    'view_hours_30d', 'completion_rate', 'active_days_30d', 'activity_rate',
    'titles_started', 'titles_abandoned', 'thumbs_up', 'my_list_saves',
    'social_engagement', 'hours_per_active_day', 'kids_content_hours',
    'num_profiles', 'has_kids', 'num_kids', 'is_mobile_only', 'is_multi_device',
    'num_devices', 'region_Europe', 'region_Latin America', 'region_North America'
]

# Filter to only existing columns
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols]
y = df['churned']

# Train/validation/test split: 70/15/15
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

In [ ]:
# Train logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

# Evaluate on validation set
val_preds = lr.predict(X_val_sc)
val_probs = lr.predict_proba(X_val_sc)[:, 1]
val_auc = roc_auc_score(y_val, val_probs)

print(f'Validation ROC-AUC: {val_auc:.4f}')
print('\nClassification Report (Validation Set):')
print(classification_report(y_val, val_preds, target_names=['Retained', 'Churned']))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_estimator(lr, X_val_sc, y_val,
                                       display_labels=['Retained', 'Churned'],
                                       cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix — Validation Set')

# Feature importances (coefficients)
coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': lr.coef_[0]})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=True).index)
coef_df.tail(12).plot(kind='barh', x='feature', y='coefficient', ax=axes[1],
                       color=['salmon' if x < 0 else 'steelblue' for x in coef_df.tail(12)['coefficient']])
axes[1].set_title('Top Feature Coefficients (Logistic Regression)')
axes[1].set_xlabel('Coefficient Value')
axes[1].legend().remove()
axes[1].axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation on test set
test_probs = lr.predict_proba(X_test_sc)[:, 1]
test_auc = roc_auc_score(y_test, test_probs)
print(f'Test ROC-AUC: {test_auc:.4f}')

## 7. Summary of Findings

From the initial EDA and baseline model:

- **Active days** is the strongest single predictor of retention. Members who open the app regularly are far less likely to churn.
- **Completion rate** (titles finished / titles started) is the next most important signal — members who consistently finish what they start have lower churn risk.
- **Mobile-only users** show a noticeably higher churn rate compared to multi-device users.
- **Households with kids** churn less, and the effect grows with more children.
- The baseline logistic regression achieves a test ROC-AUC around 0.85+, which is a solid starting point for the full model comparison in Module 24.

**Next steps (Module 24):** Compare against Ridge-regularized logistic regression, Random Forest, and Gradient Boosting. Tune hyperparameters via GridSearchCV and produce final action-oriented recommendations.